# Flu Shot Learning: Logistic Regression with Full Preprocessing

This notebook trains one regularized logistic regression model per vaccine target (Binary Relevance). It uses a complete sparse preprocessing pipeline so that missingness, categorical responses, and ordinal survey structure are all available to the linear model.

**Preprocessing:** numeric median imputation with missingness indicators, one-hot encoding for every categorical column, standardized numeric and ordinal features, and ordinal encodings for `age_group`, `education`, and `income_poverty` in parallel with their one-hot versions.

**Validation:** five-fold `StratifiedKFold` using the same joint-label stratification key as the CatBoost experiments.

**Metric:** mean ROC AUC across `h1n1_vaccine` and `seasonal_vaccine`.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

RANDOM_STATE = 2026
N_SPLITS = 5
TARGETS = ['h1n1_vaccine', 'seasonal_vaccine']
ORDINAL_COLUMNS = ['age_group', 'education', 'income_poverty']
pd.set_option('display.max_columns', 100)

## 1. Load competition data

In [2]:
def find_data_dir():
    candidates = [
        Path('/kaggle/input/datasets/barshamishra19/flu-shot-learning-dataset'),
        
    ]
    for candidate in candidates:
        if (candidate / 'training_set_features.csv').exists():
            return candidate
    raise FileNotFoundError(
        'Could not find training_set_features.csv. Set DATA_DIR to the competition data folder.'
    )

DATA_DIR = find_data_dir()
train_features = pd.read_csv(DATA_DIR / 'training_set_features.csv')
train_labels = pd.read_csv(DATA_DIR / 'training_set_labels.csv')
test_features = pd.read_csv(DATA_DIR / 'test_set_features.csv')

assert train_features['respondent_id'].is_unique
assert test_features['respondent_id'].is_unique
assert train_features['respondent_id'].equals(train_labels['respondent_id'])
assert set(train_features['respondent_id']).isdisjoint(test_features['respondent_id'])

y = train_labels[TARGETS].astype('int8').copy()
X = train_features.drop(columns=['respondent_id']).copy()
X_test = test_features.drop(columns=['respondent_id']).copy()
stratify_key = y[TARGETS].astype(str).agg('_'.join, axis=1)

print(f'Using: {DATA_DIR.resolve()}')
print(f'Train: {X.shape}; test: {X_test.shape}')
display(y.mean().rename('positive_rate').to_frame())

Using: /kaggle/input/datasets/barshamishra19/flu-shot-learning-dataset
Train: (26707, 35); test: (26708, 35)


,positive_rate
h1n1_vaccine,0.212454
seasonal_vaccine,0.465608


## 2. Define the full linear-model preprocessing pipeline

The preprocessing is learned separately inside each fold, preventing validation information from influencing imputers, category levels, or scaling. The three ordered survey columns are sent through both the regular one-hot branch and a separate ordinal branch; regularization can select whichever representation is more useful.

Numeric imputation uses `add_indicator=True` because missingness itself is predictive in this survey. `StandardScaler(with_mean=False)` preserves the sparse matrix produced by one-hot encoding.

In [3]:
categorical_columns = X.select_dtypes(include=['object', 'category']).columns.tolist()
numeric_columns = X.select_dtypes(exclude=['object', 'category']).columns.tolist()
one_hot_columns = categorical_columns

ordinal_categories = [
    ['18 - 34 Years', '35 - 44 Years', '45 - 54 Years', '55 - 64 Years', '65+ Years'],
    ['< 12 Years', '12 Years', 'Some College', 'College Graduate'],
    ['Below Poverty', '<= $75,000, Above Poverty', '> $75,000'],
]

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
    ('scaler', StandardScaler(with_mean=False)),
])
one_hot_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent', fill_value='__MISSING__')),
    ('one_hot', OneHotEncoder(handle_unknown='ignore', sparse_output=True)),
])
ordinal_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent', fill_value='__MISSING__')),
    ('ordinal', OrdinalEncoder(
        categories=ordinal_categories,
        handle_unknown='use_encoded_value',
        unknown_value=-1,
    )),
    ('scaler', StandardScaler(with_mean=False)),
])

def make_preprocessor():
    return ColumnTransformer(
        transformers=[
            ('numeric', numeric_pipeline, numeric_columns),
            ('one_hot', one_hot_pipeline, one_hot_columns),
            ('ordinal', ordinal_pipeline, ORDINAL_COLUMNS),
        ],
        remainder='drop',
        sparse_threshold=1.0,
    )

print(f'Numeric columns: {len(numeric_columns)}')
print(f'Categorical columns: {len(categorical_columns)}')
print(f'Ordinal columns included in both branches: {ORDINAL_COLUMNS}')

Numeric columns: 23
Categorical columns: 12
Ordinal columns included in both branches: ['age_group', 'education', 'income_poverty']


## 3. Five-fold target-specific logistic regression

`LogisticRegressionCV` searches a grid of regularization strengths for each target within each outer fold. The outer `StratifiedKFold` uses the joint-label key so every validation fold maintains the same four label combinations as the training data.

In [4]:
outer_splitter = StratifiedKFold(
    n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE
)
folds = list(outer_splitter.split(X, stratify_key))
inner_splitter = StratifiedKFold(
    n_splits=3, shuffle=True, random_state=RANDOM_STATE
)
regularization_grid = np.logspace(-3, 2, 8)
oof = pd.DataFrame(index=X.index, columns=TARGETS, dtype=float)
test_predictions = pd.DataFrame(0.0, index=X_test.index, columns=TARGETS)
selected_cs = {target: [] for target in TARGETS}

def make_model():
    return Pipeline([
        ('preprocessor', make_preprocessor()),
        ('classifier', LogisticRegressionCV(
            Cs=regularization_grid,
            cv=inner_splitter,
            scoring='roc_auc',
            solver='saga',
            penalty='l2',
            max_iter=3000,
            n_jobs=-1,
            refit=True,
            random_state=RANDOM_STATE,
        )),
    ])

for fold_number, (train_index, valid_index) in enumerate(folds, start=1):
    print(f'Fold {fold_number}/{N_SPLITS}')
    for target in TARGETS:
        model = make_model()
        model.fit(X.iloc[train_index], y.iloc[train_index][target])
        oof.loc[valid_index, target] = model.predict_proba(X.iloc[valid_index])[:, 1]
        test_predictions[target] += model.predict_proba(X_test)[:, 1] / N_SPLITS
        selected_cs[target].append(float(model.named_steps['classifier'].C_[0]))

auc_by_target = {
    target: roc_auc_score(y[target], oof[target])
    for target in TARGETS
}
print('\nOOF ROC AUC by target:')
display(pd.Series(auc_by_target).rename('roc_auc').to_frame())
print(f"Mean OOF ROC AUC: {np.mean(list(auc_by_target.values())):.5f}")
print('Selected C values by fold:', selected_cs)

Fold 1/5
Fold 2/5
Fold 3/5
Fold 4/5
Fold 5/5

OOF ROC AUC by target:


,roc_auc
h1n1_vaccine,0.860783
seasonal_vaccine,0.855284


Mean OOF ROC AUC: 0.85803
Selected C values by fold: {'h1n1_vaccine': [0.13894954943731375, 0.13894954943731375, 0.13894954943731375, 0.13894954943731375, 0.13894954943731375], 'seasonal_vaccine': [0.13894954943731375, 0.7196856730011522, 0.13894954943731375, 0.13894954943731375, 0.7196856730011522]}


## 4. Save OOF predictions and competition submission

Test probabilities are averaged over the five outer-fold models and remain in the original respondent order.

In [5]:
output_dir = Path.cwd() / 'outputs'
output_dir.mkdir(parents=True, exist_ok=True)

submission = pd.DataFrame({'respondent_id': test_features['respondent_id']})
submission[TARGETS] = test_predictions[TARGETS].to_numpy()
submission_path = output_dir / 'submission_logistic_regression.csv'
oof_path = output_dir / 'oof_logistic_regression.csv'
submission.to_csv(submission_path, index=False)
oof.assign(respondent_id=train_features['respondent_id']).to_csv(oof_path, index=False)

print(f'Saved submission: {submission_path.resolve()}')
print(f'Saved OOF predictions: {oof_path.resolve()}')
display(submission.head())

Saved submission: /kaggle/working/outputs/submission_logistic_regression.csv
Saved OOF predictions: /kaggle/working/outputs/oof_logistic_regression.csv


,respondent_id,h1n1_vaccine,seasonal_vaccine
0,26707,0.094827,0.339302
1,26708,0.037834,0.044035
2,26709,0.272316,0.550869
3,26710,0.576835,0.891235
4,26711,0.280665,0.529531
